In [ ]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')

In [ ]:
#load env-vars
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pgjar = os.getenv('POSTGRE_JAR')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [ ]:
#create session 
spark = (( 
          SparkSession
          .builder
          .appName('invoices')
          .config('spark.jars', f'{pgjar}')
          .config('spark.executor.extraClassPath', f'{pgjar}')
          .config('spark.driver.extraClassPath', f'{pgjar}')
          .getOrCreate()
        ))

In [ ]:
#import invoice data from sales.invoice table

tmp_invoiceline_frame = ((spark
                            .read
                            .format('jdbc')
                            .option('url', pgconn)
                            .option('driver', pgdriver)
                            .option('password', pgpwd)
                            .option('user', pguser)
                            .option('query', 
                                        
                                        """ 
                                            select
                                            
                                                invoiceid,
                                                invoicelineid,
                                                stockitemid,
                                                packagetypeid,
                                                quantity,
                                                unitprice,
                                                taxamount,
                                                extendedprice,
                                                lineprofit,
                                                taxrate,
                                                _airbyte_ab_id,
                                                _airbyte_invoicelines_hashid as invoiceline_hashid,
                                                _airbyte_emitted_at as elt_start,
                                                _airbyte_normalized_at as elt_end
                                                
                                            from sales.invoicelines

                                        """
                                    )
                        )).load().withColumnRenamed('_airbyte_ab_id', 'elt_invoiceline_hashid')

In [ ]:
tmp_invoiceorder_frame = ((spark
                           .read
                           .format('jdbc')
                           .option('driver', pgdriver)
                           .option('url', pgconn)
                           .option('user', pguser)
                           .option('password', pgpwd)
                           .option('query', 
                                    """ 
                                    
                                        select
                                        
                                            orderid,
                                            invoiceid,
                                            customerid,
                                            salespersonpersonid,
                                            accountspersonid,
                                            customerpurchaseordernumber,
                                            deliverymethodid,
                                            billtocustomerid,
                                            contactpersonid,
                                            packedbypersonid,
                                            invoicedate,
                                            cast(confirmeddeliverytime as date) as deliverydate,
                                            totaldryitems,
                                            totalchilleritems,
                                            _airbyte_ab_id,
                                            _airbyte_invoices_hashid as invoices_hashid
                                            
                                        from sales.invoices
                                    
                                    """
                                   )
                        )).load().withColumnRenamed('_airbyte_ab_id', 'elt_invoices_hashid')

In [ ]:
#joining dataframes 

join_invoiceframes = (( tmp_invoiceline_frame
                        .join(tmp_invoiceorder_frame, 'invoiceid', 'left')
                            
                        ))

In [108]:
invoice = (( join_invoiceframes
                .select('invoiceid',
                        'invoicelineid',
                        'orderid',
                        'stockitemid',
                        'customerid',
                        'salespersonpersonid',
                        'accountspersonid',
                        'customerpurchaseordernumber',
                        'billtocustomerid',
                        'contactpersonid',
                        'packedbypersonid',
                        'packagetypeid',
                        'quantity',
                        'unitprice',
                        'taxamount',
                        'extendedprice',
                        'lineprofit',
                        'taxrate',
                        'totaldryitems',
                        'totalchilleritems',
                        'elt_invoices_hashid',
                        'elt_invoiceline_hashid',
                        'invoices_hashid',
                        'invoiceline_hashid',
                        'elt_start',
                        'elt_end'
                        )
                .withColumnRenamed('salespersonpersonid', 'salespersonid')
                .withColumnRenamed('customerpurchaseordernumber', 'purchaseordernumber')
        ))



In [109]:
invoice.createOrReplaceTempView('temp')


invoice = spark.sql("""
                select cast(unitprice as numeric(12,2)) as unitprirce
                from temp
                
                """)

In [110]:
((
    invoice
        .write
        .format('jdbc')
        .option('url', pgconn)
        .option('driver', pgdriver)
        .option('user', pguser)
        .option('password', pgpwd)
        .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))